# Mühlenanwendungsfall — Drift-Erkennung

Wendet die fünf Detektoren aus `drift_detection.py` — KSWIN, ADWIN, DDM, EDDM und den Schwellwert-Detektor RMSE — auf den normierten Modellfehler des 1-min-Stroms an. Aufbau und Zielfunktion sind die von *syn_detection* und *tep_detection*.

**Ereignisreferenz statt Ground Truth.** An der realen Anlage ist die Driftursache nicht gemessen. An ihre Stelle treten die Kalibrierungen aus *mill_context*, abgebildet auf den ersten Punkt des Stroms am oder nach ihrem Zeitpunkt.

**Zeitachse in Tagen.** Toleranz, Refraktärzeit und die Grenzen des Suchraums sind in Tagen festgelegt und werden über `mio.days_to_points` in Punkte umgerechnet. Die Grenzen sind die des synthetischen Falls, als Zeitspannen gelesen.

In [ ]:
FORCE_RECOMPUTE = False
IS_FINAL = True
RUN_PHASE = "full"
TEST_MODE = False

## Konstanten

`TOLERANCE_REL` und `COOLDOWN_REL` sind die relativen Vorgaben der beiden anderen Fälle: die Toleranz als Anteil des mittleren Ereignisabstands, die Refraktärzeit der Darstellung als Anteil der Stromlänge. Beide werden hier in Tagen gebildet und auf `QUANT_DAYS` gerundet.

`DET_SPACE_DAYS` ersetzt die Grenzen der Zählparameter in `dd._suggest_params`. Die Werte sind die Punktgrenzen des synthetischen Falls mal dessen Tuningraster von fünf Minuten. `DET_SPACE_MIN_POINTS` hält die Untergrenzen, die aus dem Verfahren selbst folgen (Stichprobenumfang des KS-Tests, Anlauf der Fehlerrate) und sich nicht in Zeit ausdrücken lassen. `TEST_MODE` verkürzt Strom und Budget für einen Probelauf und schreibt unter eigenem Präfix.

In [ ]:
RUN_ID = None          # data_scored_1min_stream aus mill_model; None -> neuestes
CONTEXT_ID = None      # context_events aus mill_context; None -> neuestes
NB_NAME = "mill_detection"
SEED = 42

DETECTORS = ("KSWIN", "ADWIN", "DDM", "EDDM", "RMSE")
N_TRIALS = {"KSWIN": 40, "ADWIN": 40, "RMSE": 40, "DDM": 200, "EDDM": 200}
DOWNSAMPLE = 5
WEIGHTS = (3.0, 0.3, 1.5)
FA_DECADES = 3.0
TOP_K = 5

TOLERANCE_REL = 0.0092
COOLDOWN_REL = 0.0095
QUANT_DAYS = 0.05

MINUTE = 1.0 / 1440.0
DET_SPACE_DAYS = {
    "KSWIN": {"min_num_instances": (250 * MINUTE, 10000 * MINUTE)},
    "ADWIN": {"clock": (5 * MINUTE, 320 * MINUTE),
              "min_window_size": (25 * MINUTE, 320 * MINUTE)},
    "DDM": {"min_num_instances": (150 * MINUTE, 2500 * MINUTE)},
    "EDDM": {"min_num_misclassified_instances": (150 * MINUTE, 2500 * MINUTE)},
    "RMSE": {"window": (75 * MINUTE, 5000 * MINUTE)},
}
DET_SPACE_MIN_POINTS = {
    "KSWIN": {"min_num_instances": 40},
    "ADWIN": {"clock": 1, "min_window_size": 5},
    "DDM": {"min_num_instances": 30},
    "EDDM": {"min_num_misclassified_instances": 30},
    "RMSE": {"window": 15},
}

PLOT_FRAC = 0.01
TEST_SPAN_DAYS = 120
if TEST_MODE:
    IS_FINAL = False
    N_TRIALS = {d: 4 for d in DETECTORS}
    TOP_K = 2

In [ ]:
import os
from pathlib import Path

work_dir = os.getcwd()
DEFAULT_BASE_DIR = os.path.normpath(os.path.join(work_dir, "..", ".."))

base_dir = Path(os.environ.get("BASE_DIR", DEFAULT_BASE_DIR)).resolve()
data_dir = base_dir / "data" / "mill"
model_dir = base_dir / "models"
plot_dir = base_dir / "plots"
results_dir = base_dir / "results"
for _d in (data_dir, model_dir, plot_dir, results_dir):
    os.makedirs(_d, exist_ok=True)

## Fehlerstrom und Ereignisse

`stream_cfg` ist der gemeinsame Kopf aller Cache-Schlüssel: die Kennung des Modells mit der des 1-min-Stroms. Ereignisse, deren Toleranzfenster über das Stromende reicht, zählen nicht.

In [ ]:
import numpy as np
import pandas as pd
from src.utils import run_registry as rr
from src.utils import mill_io as mio
from src.utils import mill_drift as md
from src.utils.run_status import RunStatus

rr.assert_current()
PREFIX = "milltest" if TEST_MODE else "mill"
data_store = rr.mill_data_store(data_dir)
model_store = rr.mill_model_store(model_dir)
run_store = rr.ArtifactStore(model_dir, PREFIX)
status = RunStatus(NB_NAME, base_dir / "_temp" / "status")

run_id = RUN_ID or data_store.latest_id("data_scored_1min_stream")
if run_id is None:
    raise FileNotFoundError("Kein data_scored_1min_stream -- bitte mill_model.ipynb ausfuehren.")
scored = data_store.load("data_scored_1min_stream", run_id=run_id)
stream_cfg = rr.RunConfig.from_document(scored.attrs["config"])
MODEL_ID = stream_cfg.base_id

ctx_id = CONTEXT_ID or model_store.latest_id("context_events")
if ctx_id is None:
    raise FileNotFoundError("Kein context_events -- bitte mill_context.ipynb ausfuehren.")
events = model_store.load("context_events", run_id=ctx_id)

if TEST_MODE:
    scored = scored.loc[:scored.index[0] + pd.Timedelta(days=TEST_SPAN_DAYS)]

ledger = None if TEST_MODE else rr.run_ledger(work_dir)
if ledger is not None:
    ledger.bind("detection", parents={"model": MODEL_ID, "context": ctx_id})
plot_store = rr.mill_plot_store(plot_dir, prefix=PREFIX, ledger=ledger)
res_store = rr.mill_results_store(results_dir, prefix=PREFIX, ledger=ledger)

error_stream = np.nan_to_num(scored["model_error_norm"].to_numpy(float))
time_index = scored.index
N = len(error_stream)
SPAN_DAYS = (time_index[-1] - time_index[0]).total_seconds() / 86400.0
PPD = mio.points_per_day(time_index)
print(f"stream_id={stream_cfg.id}  model_id={MODEL_ID}  context_id={ctx_id}")
print(f"Fehlerstrom: {N} Punkte, {PPD:.0f} je Kalendertag, "
      f"{time_index[0]:%d.%m.%Y} bis {time_index[-1]:%d.%m.%Y}")

In [ ]:
def pts(days, index=None):
    return mio.days_to_points(days, time_index if index is None else index)


def _quant(days):
    return float(max(QUANT_DAYS, round(days / QUANT_DAYS) * QUANT_DAYS))


_times = pd.DatetimeIndex(events["timestamps"])
_times = _times[(_times > time_index[0]) & (_times < time_index[-1])]
EVENT_SPACING_DAYS = (float(np.mean(np.diff(_times.values) / np.timedelta64(1, "D")))
                      if len(_times) > 1 else SPAN_DAYS)
TOLERANCE_DAYS = _quant(TOLERANCE_REL * EVENT_SPACING_DAYS)
COOLDOWN_DAYS = _quant(COOLDOWN_REL * SPAN_DAYS)
TOLERANCE = pts(TOLERANCE_DAYS)
COOLDOWN = pts(COOLDOWN_DAYS)

_idx = time_index.searchsorted(_times)
_keep = _idx + TOLERANCE < N
event_times = _times[_keep]
sudden_idx = sorted({int(i) for i in _idx[_keep]})
drift_active = np.zeros(N, bool)   # kein gemessener Driftzustand

print(f"{len(sudden_idx)} Ereignisse ({events['event_set']}), mittlerer Abstand "
      f"{EVENT_SPACING_DAYS:.0f} Tage")
print(f"TOLERANCE = {TOLERANCE_DAYS:.2f} Tage = {TOLERANCE} Punkte | "
      f"COOLDOWN = {COOLDOWN_DAYS:.2f} Tage = {COOLDOWN} Punkte")

## Suchraum

`SPACE_ID` ist die Kennung von `dd._suggest_params` aus dem geladenen Code-Objekt, unverändert gegenüber Kapitel 3. `DET_BOUNDS` sind die Grenzen in Punkten des Tuningrasters; sie stehen zusammen mit ihren Vorgaben in Tagen im Cache-Schlüssel.

In [ ]:
import drift_detection as dd

dd.assert_current()
SPACE_ID = dd.search_space_id()

tune_index = time_index[::DOWNSAMPLE]
DET_BOUNDS = {d: md.points_bounds(DET_SPACE_DAYS[d], lambda x: pts(x, tune_index),
                                  DET_SPACE_MIN_POINTS[d]) for d in DETECTORS}
print(f"drift_detection {dd.MODULE_SHA} | Suchraum {SPACE_ID}")
for _d in DETECTORS:
    for _k, (_lo, _hi) in DET_BOUNDS[_d].items():
        _a, _b = DET_SPACE_DAYS[_d][_k]
        print(f"{_d:6s} {_k:32s} {_a:6.3f} .. {_b:6.3f} Tage = {_lo:5d} .. {_hi:5d} Punkte")

## Parameter-Tuning

Getunt wird auf jedem `DOWNSAMPLE`-ten Punkt, je Detektor mit eigenem Budget. Der Cache-Fall `detect_tuning` wird exakt getroffen.

In [ ]:
from types import SimpleNamespace
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

det_cfg = stream_cfg.compose(detector={
    "detectors": DETECTORS, "n_trials": N_TRIALS, "downsample": DOWNSAMPLE,
    "weights": list(WEIGHTS), "fa_decades": FA_DECADES, "space": SPACE_ID,
    "tolerance_days": TOLERANCE_DAYS, "tolerance": TOLERANCE,
    "cooldown_days": COOLDOWN_DAYS, "cooldown": COOLDOWN,
    "bounds_days": DET_SPACE_DAYS, "bounds": DET_BOUNDS,
    "context_id": ctx_id, "events": [str(t) for t in event_times], "seed": SEED,
    **({"test_span_days": TEST_SPAN_DAYS} if TEST_MODE else {})})
print(f"det_cfg = {det_cfg.id}")

_tuning_cached = run_store.exists("detect_tuning", det_cfg) and not FORCE_RECOMPUTE
_tuning = (run_store.load("detect_tuning", det_cfg) if _tuning_cached else
           {"best_params": {}, "user_attrs": {}, "best_value": {}, "top_params": {}})

_retuned = False
studies, best_params = {}, {}
for _name in DETECTORS:
    if _name in _tuning["best_params"]:
        best_params[_name] = _tuning["best_params"][_name]
        studies[_name] = SimpleNamespace(
            best_value=_tuning["best_value"][_name],
            best_trial=SimpleNamespace(user_attrs=_tuning["user_attrs"][_name]))
        print(f"{_name:6s} aus Cache")
        continue
    status.stage(f"Tuning {_name}", total=N_TRIALS[_name])
    _study = md.tune_detector(
        _name, error_stream, sudden_idx, drift_active, tolerance=TOLERANCE,
        bounds=DET_BOUNDS[_name], n_trials=N_TRIALS[_name], downsample=DOWNSAMPLE,
        weights=WEIGHTS, fa_decades=FA_DECADES, seed=SEED,
        callbacks=[status.optuna_callback()])
    studies[_name] = _study
    best_params[_name] = _study.best_params
    _tuning["best_params"][_name] = best_params[_name]
    _tuning["user_attrs"][_name] = dict(_study.best_trial.user_attrs)
    _tuning["best_value"][_name] = float(_study.best_value)
    _tuning["top_params"][_name] = [
        dict(t.params) for t in sorted(
            (t for t in _study.trials if t.value is not None),
            key=lambda t: t.value)[:TOP_K]]
    _retuned = True
    status.done(f"score={_study.best_value:.3f}")
    run_store.save(_tuning, "detect_tuning", det_cfg)

for _name in DETECTORS:
    _a = studies[_name].best_trial.user_attrs
    print(f"{_name:6s} score={studies[_name].best_value:.3f} | recall={_a['recall']:.2f} "
          f"prec={_a['precision']:.2f} delay={_a['mean_delay']:.0f} "
          f"false={_a['n_false']:5d} -> {best_params[_name]}")

### Gegenprüfung auf dem vollen Raster

Die `TOP_K` besten Trials je Detektor laufen über den vollständigen Strom, gewählt wird der beste Score dort. Die Lage jedes gewählten Werts in seinem Intervall zeigt, ob eine Grenze bindet (ab 0,97 oder bis 0,03).

In [ ]:
def _run(name, params, cooldown=0):
    p = dict(params)
    et = p.pop("err_threshold", 0.5)
    return dd.run_detector(name, dd.build_detector(name, **p), error_stream,
                           err_threshold=et, cooldown=cooldown)


def _metrics(drifts):
    return dd.score_detections(drifts, sudden_idx, drift_active, tolerance=TOLERANCE,
                               n=N, weights=WEIGHTS, fa_decades=FA_DECADES)


cc_cfg = det_cfg.compose(crosscheck={"top_k": TOP_K, "candidates": _tuning["top_params"]})

if run_store.exists("detect_crosscheck", cc_cfg) and not FORCE_RECOMPUTE:
    crosscheck = run_store.load("detect_crosscheck", cc_cfg)
    print("Gegenpruefung aus Cache")
else:
    crosscheck = {}
    status.stage("Gegenpruefung", total=sum(len(_tuning["top_params"][d]) for d in DETECTORS))
    for _name in DETECTORS:
        _rows = []
        for _rank, _cand in enumerate(_tuning["top_params"][_name], start=1):
            _m = _metrics(_run(_name, _cand)[0])
            _rows.append({"rank_tuning": _rank, "params": dict(_cand),
                          "score": _m["score"], "recall": _m["recall"],
                          "fa_term": _m["fa_term"], "n_detected": _m["n_detected"]})
            status.step(f"{_name} Rang {_rank}")
        crosscheck[_name] = {"chosen": min(_rows, key=lambda r: r["score"]),
                             "candidates": _rows}
    run_store.save(crosscheck, "detect_crosscheck", cc_cfg)
    status.done()

best_params = {_name: crosscheck[_name]["chosen"]["params"] for _name in DETECTORS}
bounds_position = {_name: md.bounds_position(best_params[_name], DET_BOUNDS[_name])
                   for _name in DETECTORS}
for _name in DETECTORS:
    _c = crosscheck[_name]["chosen"]
    print(f"{_name:6s} Rang {_c['rank_tuning']} von {len(crosscheck[_name]['candidates'])} | "
          f"score={_c['score']:.3f} recall={_c['recall']:.2f} n_det={_c['n_detected']:6d} "
          f"-> {_c['params']}")
    for _k, _v in bounds_position[_name].items():
        _flag = "  <- Grenze bindet" if _v >= 0.97 or _v <= 0.03 else ""
        print(f"       {_k:32s} = {best_params[_name][_k] / PPD * DOWNSAMPLE:6.3f} Tage, "
              f"Lage {_v:.3f}{_flag}")

## Detektionen

`tuned` sind die Detektionen der gewählten Parameter auf dem vollen Strom, `plot_thinned` dieselben mit der Refraktärzeit der Darstellung.

In [ ]:
det_run = cc_cfg
if run_store.exists("detections", det_run) and not FORCE_RECOMPUTE and not _retuned:
    _cache = run_store.load("detections", det_run)
    tuned, plot_thinned = _cache["tuned"], _cache["plot_thinned"]
    print("Detektionen aus Cache")
else:
    status.stage("Detektionen", total=2 * len(DETECTORS))
    tuned, plot_thinned = {}, {}
    for _name in DETECTORS:
        tuned[_name] = _run(_name, best_params[_name])
        status.step(_name)
        plot_thinned[_name] = _run(_name, best_params[_name], cooldown=COOLDOWN)
        status.step(f"{_name} ausgeduennt")
    run_store.save({"tuned": tuned, "plot_thinned": plot_thinned}, "detections", det_run)
    status.done()

for _name in DETECTORS:
    print(f"{_name:6s}: {len(tuned[_name][0]):6d} Detektionen, "
          f"{len(plot_thinned[_name][0]):4d} mit Refraktaerzeit {COOLDOWN_DAYS:.1f} Tage")

## Kennzahlen

`chance` ist der Recall einer blind mit derselben Rate feuernden Folge. Nur der Abstand dazu ist eine Aussage.

In [ ]:
def _chance(n_det):
    if not n_det or not sudden_idx:
        return {"recall": 0.0, "mean_delay": None}
    return {"recall": float(1.0 - np.exp(-n_det * TOLERANCE / N)),
            "mean_delay": float(N / (2 * n_det))}


detector_results, _rows = {}, {}
for _name in DETECTORS:
    _ua = studies[_name].best_trial.user_attrs
    _drifts = tuned[_name][0]
    _m = _metrics(_drifts)
    detector_results[_name] = {
        "best_params": best_params[_name],
        "bounds_position": bounds_position[_name],
        "tuning_metrics": {_k: _ua.get(_k) for _k in
                           ["recall", "precision", "fa_term", "mean_delay", "n_false",
                            "fa_per_true", "n_sudden", "n_detected", "score"]},
        "tuning_score": float(studies[_name].best_value),
        "full_stream": {_k: _m[_k] for _k in
                        ["n_detected", "recall", "precision", "fa_term", "mean_delay",
                         "n_false", "score"]},
        "chance": _chance(_m["n_detected"]),
        "detections_per_month": float(30.0 * _m["n_detected"] / SPAN_DAYS),
        "crosscheck": {"top_k": TOP_K,
                       "rank_tuning": crosscheck[_name]["chosen"]["rank_tuning"],
                       "candidates": crosscheck[_name]["candidates"]},
        "plot_thinned": {"n_shown": len(plot_thinned[_name][0])},
        "drift_times": [str(time_index[d]) for d in _drifts],
    }
    _rows[_name] = {"n_detected": _m["n_detected"], "recall": _m["recall"],
                    "chance": detector_results[_name]["chance"]["recall"],
                    "precision": _m["precision"], "delay_days": _m["mean_delay"] / PPD,
                    "n_false": _m["n_false"], "score": _m["score"],
                    "per_month": detector_results[_name]["detections_per_month"]}
metrics = pd.DataFrame(_rows).T
print(metrics.to_string(float_format=lambda v: f"{v:.3g}"))

## Abbildungen

Je Detektor der normierte Modellfehler mit den Detektionen. Wo die Abbildungen aus Kapitel 3 das Driftsignal zeigen, stehen hier die Ereignisse der Referenz als senkrechte Linien.

In [ ]:
import matplotlib.pyplot as plt
from src.utils import thesis_style as ts

rng_plot = np.random.default_rng(SEED)
for _j, _name in enumerate(DETECTORS):
    fig, ax, ax2 = ts.error_timeseries(
        time_index,
        [(error_stream, "drift_afflicted", "Norm. Modellfehler")],
        cd=None,
        detections=[time_index[d] for d in plot_thinned[_name][0]],
        xlabel=ts.TIME_LABEL if _j == len(DETECTORS) - 1 else None,
        ylim=(-1e2, 1e2), frac=1.0 if TEST_MODE else PLOT_FRAC, rng=rng_plot,
        height_scale=0.4, month_interval=6, date_format="%b %y",
        legend=dict(bbox_to_anchor=[0.49, 1.15], loc="upper center", ncols=3)
        if _j == 0 else False,
    )
    for _t in event_times:
        ax.axvline(_t, **ts.vline("drift_event", lw=1.2))
    fig.tight_layout()
    plot_store.save_figure(fig, f"detection_{_name.lower()}", det_run, final=IS_FINAL,
                           savefig_kwargs={"dpi": 300}, archive_kwargs={"dpi": 300})
    plt.show()

## Ergebnisse speichern

In [ ]:
from src.utils import results_export as rx

(rx.ResultDoc()
 .integer("n_samples", N)
 .num("points_per_day", PPD, 1)
 .num("span_days", SPAN_DAYS, 0)
 .set("event_set", events["event_set"])
 .integer("n_events", len(sudden_idx))
 .set("event_times", [str(t) for t in event_times])
 .set("event_indices", sudden_idx)
 .num("event_spacing_days", EVENT_SPACING_DAYS, 1)
 .set("tolerance_rel", TOLERANCE_REL)
 .num("tolerance_days", TOLERANCE_DAYS, 2)
 .integer("tolerance", TOLERANCE)
 .set("cooldown_rel", COOLDOWN_REL)
 .num("cooldown_days", COOLDOWN_DAYS, 2)
 .integer("plot_cooldown", COOLDOWN)
 .set("tuning", {"n_trials": N_TRIALS, "downsample": DOWNSAMPLE,
                 "weights": list(WEIGHTS), "fa_decades": FA_DECADES, "top_k": TOP_K,
                 "space": SPACE_ID, "bounds_days": DET_SPACE_DAYS, "bounds": DET_BOUNDS})
 .set("detectors", detector_results)
 .save(res_store, "detection", det_run, final=IS_FINAL))

In [ ]:
from src.utils import latex_export as lx

mx = lx.MacroExport("automatisch erzeugt aus mill_detection.ipynb - nicht manuell editieren")
mx.comment("Zeitachse (in Tagen festgelegt)")
mx.num("millDriftToleranceDays", TOLERANCE_DAYS, 2)
mx.num("millDriftToleranceH", 24 * TOLERANCE_DAYS, 0)
mx.integer("millDriftTolerance", TOLERANCE)
mx.num("millDriftCooldownDays", COOLDOWN_DAYS, 1)
mx.integer("millDriftNEvents", len(sudden_idx))
mx.num("millDriftEventSpacingDays", EVENT_SPACING_DAYS, 0)
mx.integer("millDriftDownsample", DOWNSAMPLE)
mx.integer("millDriftTopK", TOP_K)

mx.comment("Detektionskennzahlen je Detektor")
for _name in DETECTORS:
    _r = metrics.loc[_name]
    mx.num(f"millDrift{_name}Recall", float(_r["recall"]), 2)
    mx.num(f"millDrift{_name}Chance", float(_r["chance"]), 2)
    mx.num(f"millDrift{_name}Precision", float(_r["precision"]), 2)
    mx.num(f"millDrift{_name}DelayH", 24 * float(_r["delay_days"]), 1)
    mx.integer(f"millDrift{_name}Drifts", int(_r["n_detected"]))
    mx.integer(f"millDrift{_name}False", int(_r["n_false"]))
    mx.num(f"millDrift{_name}PerMonth", float(_r["per_month"]), 1)
    mx.integer(f"millDrift{_name}DriftsPlot", len(plot_thinned[_name][0]))
mx.save(res_store, "detection", det_run, final=IS_FINAL)

In [ ]:
print(f"det_run.id = {det_run.id}")
status.finish("\n".join(f"{d}: Recall {metrics.loc[d, 'recall']:.2f} "
                         f"(Zufall {metrics.loc[d, 'chance']:.2f}), "
                         f"{int(metrics.loc[d, 'n_detected'])} Detektionen" for d in DETECTORS))